In [2]:
import dspy

# Konfiguration des lokalen Sprachmodells
local_llm = dspy.LM(
    "openai/Qwen3-VL-8B-Instruct-Q4_K_M.gguf", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=0.1,
    cache=False
)

dspy.configure(lm=local_llm)

In [3]:
class GenerateKeywords(dspy.Signature):
    """Generiert eine Liste von 5-7 relevanten Schlüsselwörtern aus einem langen Text."""
    long_text = dspy.InputField(desc="Ein langer Text, aus dem Schlüsselwörter extrahiert werden sollen.")
    keywords = dspy.OutputField(desc="Eine durch Kommas getrennte Liste von 5-7 Schlüsselwörtern.")

class SummarizeWithKeywords(dspy.Signature):
    """Erstellt eine kurze Zusammenfassung eines Textes unter Berücksichtigung der vorgegebenen Schlüsselwörter."""
    long_text = dspy.InputField(desc="Der Originaltext, der zusammengefasst werden soll.")
    keywords = dspy.InputField(desc="Relevante Schlüsselwörter, auf die sich die Zusammenfassung konzentrieren soll.")
    summary = dspy.OutputField(desc="Eine prägnante Zusammenfassung in 2-3 Sätzen.")

In [4]:
class SummarizationModule(dspy.Module):
    def __init__(self):
        super().__init__()
        # 1. Deklaration der Sub-Module im Konstruktor
        self.keyword_generator = dspy.Predict(GenerateKeywords)
        self.summarizer = dspy.Predict(SummarizeWithKeywords)

    def forward(self, long_text):
        # 2. Definition des Kontrollflusses in der forward-Methode
        # Schritt 1: Schlüsselwörter generieren
        keywords_prediction = self.keyword_generator(long_text=long_text)

        # Schritt 2: Zusammenfassung mit den generierten Schlüsselwörtern erstellen
        summary_prediction = self.summarizer(long_text=long_text, keywords=keywords_prediction.keywords)

        # Rückgabe der finalen Ergebnisse
        return dspy.Prediction(
            keywords=keywords_prediction.keywords,
            summary=summary_prediction.summary
        )

In [6]:
# Beispieltext zur Zusammenfassung
example_text = """
Die künstliche Intelligenz (KI) hat sich in den letzten Jahren rasant entwickelt.
Besonders große Sprachmodelle (LLMs) wie GPT-4 haben die Fähigkeiten von Maschinen,
menschliche Sprache zu verstehen und zu generieren, revolutioniert. Diese Modelle werden auf
riesigen Textmengen trainiert und können für eine Vielzahl von Aufgaben eingesetzt werden,
darunter Textzusammenfassung, Übersetzung, Beantwortung von Fragen und sogar das Schreiben
von kreativen Texten oder Code. Die zugrunde liegende Architektur, bekannt als Transformer,
ist entscheidend für diesen Erfolg, da sie es den Modellen ermöglicht, Kontexte und
Beziehungen zwischen Wörtern in langen Textsequenzen effektiv zu verarbeiten.
"""

# Instanziierung und Aufruf des Moduls
summarizer_module = SummarizationModule()
result = summarizer_module(long_text=example_text)

# Ausgabe der Ergebnisse
print(f"Generierte Schlüsselwörter: {result.keywords}")
print(f"Erstellte Zusammenfassung: {result.summary}")

Generierte Schlüsselwörter: künstliche Intelligenz, Sprachmodelle, GPT-4, Transformer, Textgenerierung, KI-Entwicklung, maschinelles Lernen
Erstellte Zusammenfassung: Künstliche Intelligenz, insbesondere große Sprachmodelle wie GPT-4, hat durch ihre Fähigkeit zur Textgenerierung und -verarbeitung die KI-Entwicklung revolutioniert. Auf Basis der Transformer-Architektur können diese Modelle auf riesigen Datensätzen trainiert werden und vielfältige Aufgaben wie Übersetzung, Fragebeantwortung oder kreatives Schreiben übernehmen.
